# Arrhythmia Classification (ECG) — Akida Benchmark

<p align="right">
Run Time: ~10 minutes
</p>

This notebook evaluates and benchmarks the ECG beat classifier on Akida 1 hardware.

For details of the dataset and the preparation of the model, see the neighbouring
[README.md](README.md) and
[arrhythmia_notebook_training.ipynb](arrhythmia_notebook_training.ipynb).

> **Note:** the hardware sections below need a physical AKD1500 device, and the power
> measurements read an INA219 over I2C. Everything is guarded, so the notebook runs
> end-to-end without a board — the hardware cells simply skip. It will not produce
> hardware numbers in Colab.

## Model

A pretrained Akida model is included in this repo, at
`pretrained_models/arrhythmia_classification_qat.fbz`. As with all model files here,
it is handled via `git-lfs` (for efficient large file storage). If you have not set
that up yet, see the [Trained models](../../../README.md#trained-models) section of
the top-level README.

If you have run through the training scripts or notebook and would rather benchmark
the model that produced, simply change `MODELS_DIR` and `MODEL_FILENAME` in the
following cell to point at `./models/`.

In [ ]:
import os
import sys
import time

import akida
import numpy as np

from sklearn.metrics import classification_report
from tqdm import tqdm

os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '1')

# Repo root on sys.path so `brainchip_utils` imports work even when the package
# has not been pip-installed.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

DATA_PATH = './data/mitdb'
MODELS_DIR = './pretrained_models/'
MODEL_FILENAME = 'arrhythmia_classification_qat.fbz'
akida_model_path = os.path.join(MODELS_DIR, MODEL_FILENAME)

BATCH_SIZE = 64

# Beats used for sparsity and every benchmark below. Matches the count used by
# arrhythmia_benchmark.py, so the figures line up with the README tables.
NUM_SAMPLES = 1000

# 7 is the seed the reported figures come from, and it selects the benchmark
# sample draw.
SEED = 7

To load the model, we simply pass the path to the `.fbz` file to `akida.Model()`:

In [ ]:
# Load the Akida model
akida_model = akida.Model(akida_model_path)
akida_model.summary()

## Dataset

Evaluation runs on the **inter-patient test split** — the DS2 records. None of those
22 patients appear anywhere in the training data, which is what makes this the number
worth quoting.

Each beat arrives as a 36 × 32 single-channel image: a 32 × 32 Morlet wavelet
scalogram with four standardised RR-interval feature rows appended underneath. See
the [README](README.md#dataset) for how that is constructed.

> **First run:** if the MIT-BIH records are not already present, they are downloaded
> from PhysioNet (~100 MB) and the scalogram cache is built. That takes about a
> minute, and happens only once.

In [ ]:
from arrhythmia_data import INPUT_SHAPE, TARGET_NAMES, get_samples, get_test_data

test_ds = get_test_data(DATA_PATH, INPUT_SHAPE, batch_size=BATCH_SIZE)

## Evaluation of Akida Model

We now run evaluation through the Akida model, to check that accuracy matches what
the quantized Keras model achieved. If an Akida 1 hardware device is connected it
will be used for inference; if not, the code falls back to the software backend,
which delivers a bit-accurate simulation of the hardware result. Let's run that check
before going any further.

### Check for a connected Akida hardware device

The `akida.devices()` function detects connected hardware. It returns a list — if
it's empty, there were no devices. Typically only a single Akida device is connected
on a given machine, so we can just take the first one.

In [ ]:
devices = akida.devices()
if len(devices) > 0:
    # Hardware is available
    device = devices[0]
else:
    # Hardware is not available
    device = None

In the present case we want to be a bit more careful, and check that the device is
the right version for the model we are testing (here, Akida IP version 1). We import
a local helper to do that — check out
[brainchip_utils/hardware_utils.py](../../../brainchip_utils/hardware_utils.py) if
you are interested in the details.

Having found a matching device, `map` schedules the model onto it. `MapMode.AllNps`
spreads the model across all available Neural Processors for maximum parallelism, and
`hw_only=True` refuses to fall back to software for any layer — so if this succeeds,
the whole network really is running on the chip.

In [ ]:
from brainchip_utils.hardware_utils import get_akida_device

# Look for a matching hardware device
device = get_akida_device(target_version=akida_model.ip_version)
if device is not None:
    akida_model.map(device, mode=akida.MapMode.AllNps, hw_only=True)

### Run Evaluation on Akida

The Akida runtime cannot consume `tf.data.Dataset` objects directly — it expects a 4D
numpy array `(n, h, w, c)` in uint8 — so we iterate over the test batches manually.
That situation arises here only because the dataset is delivered in TensorFlow format;
in a real deployment, beats would be extracted from the ECG stream and sent to Akida
without ever passing through a `tf.data` pipeline.

One detail specific to this example: the dataset yields **float32 in the range
[0, 255]**, not uint8. The values are integral, so the cast back to `uint8` for the
Akida call is exact and lossless — but it does have to be done.

The model output tensor has shape `(B, 1, 1, C)`, squeezed to `(B, C)` before taking
the class argmax.

Note that accuracy alone flatters any model on this dataset: around 90% of the test
beats are normal, so predicting "normal" unconditionally would score 0.90. The
per-class breakdown is the honest summary, and the supraventricular (S) row is the
hard one — for reasons the README's
[Dataset limits](README.md#dataset-limits) section works through in full.

In [ ]:
labels_all = []
logits_all = []
for image_batch, label_batch in tqdm(test_ds, desc='Evaluating on Akida'):
    # Datasets here yield float32 in [0, 255]; Akida wants uint8. Values are
    # integral, so the cast is exact.
    logits_batch = akida_model.predict(image_batch.numpy().astype(np.uint8))
    logits_batch = logits_batch.squeeze(axis=(1, 2))  # (B, 1, 1, C) -> (B, C)

    labels_all.append(label_batch.numpy())
    logits_all.append(logits_batch)

labels_all = np.concatenate(labels_all)
preds = np.argmax(np.concatenate(logits_all), axis=1)

akida_acc = float(np.mean(preds == labels_all))
scores = classification_report(labels_all, preds, target_names=list(TARGET_NAMES),
                               output_dict=True, zero_division=0)
print(f'Akida accuracy: {akida_acc:.4f}    '
      f'Macro F1: {scores["macro avg"]["f1-score"]:.4f}\n')
print(classification_report(labels_all, preds, target_names=list(TARGET_NAMES),
                            digits=4, zero_division=0))

### Activation Sparsity

Akida hardware skips computation for zero-valued activations, so activation sparsity
directly reduces both energy consumption and inference latency. Below we measure
per-layer sparsity over 1000 real beats.

The `samples` array built here is also what feeds every benchmark in the rest of the
notebook — see the note in the next section on why that matters.

In [ ]:
from akida_models.sparsity import compute_sparsity
from brainchip_utils.plot_utils import pretty_print_sparsity

samples = get_samples(DATA_PATH, INPUT_SHAPE, num_samples=NUM_SAMPLES, seed=SEED)
sparsity_dict = compute_sparsity(akida_model, samples=samples)
pretty_print_sparsity(sparsity_dict)

print(f'\nMean activation sparsity over {NUM_SAMPLES} beats: '
      f'{np.mean(list(sparsity_dict.values())) * 100:.2f}%')

## Hardware Benchmark

**These cells require a physical AKD1500 device to be connected.** If `device is None`
(reported in the evaluation section above), they will all skip cleanly and you can
read through without running anything.

Akida is an event-driven architecture: computation scales with the number of non-zero
activations, not with tensor size. That makes benchmark results *input-dependent* —
random or synthetic data would give artificially fast or slow timings, because it
would carry activation statistics no real ECG beat has. The `samples` array loaded
above holds real beats, and is therefore the correct input to benchmark with.

### Simple Benchmark

The simplest way to time an Akida model is to call `forward` in a loop and read back
two clocks after each inference:

- **System clock** (`time.perf_counter_ns`) — wall time, including Python overhead
  and the USB/PCIe transfer.
- **On-chip clock** (`akida_model.metrics['inference_clk']`) — raw clock cycles
  counted by the AKD1500 itself. Dividing by the 400 MHz core frequency gives the
  pure compute time.

The two numbers should agree closely; a large divergence would point to a transfer or
driver bottleneck rather than a slow model.

Here we map in `MapMode.Minimal`, which schedules the model onto the fewest Neural
Processors it needs. The first inference on a freshly mapped model includes loading
the weights to the device, so a priming frame is run and discarded before timing
starts.

In [ ]:
CLOCK_FREQUENCY = 400e6  # 400 MHz for AKD1500

if device is not None:
    akida_model.map(device, mode=akida.MapMode.Minimal, hw_only=True)

    # Run a priming frame, so that the model is loaded to the device
    # and the subsequent benchmarking reflects inference time only
    akida_model.forward(samples[:2])

    inf_clks = []
    inf_times = []
    for i in range(len(samples)):
        start_t = time.perf_counter_ns()
        akida_model.forward(samples[i:i + 1], batch_size=1)
        inf_times.append(time.perf_counter_ns() - start_t)
        # Number of on-device clock cycles for that inference
        inf_clks.append(akida_model.metrics['inference_clk'])

    mean_inf_clk = np.mean(inf_clks) / CLOCK_FREQUENCY * 1e3  # cycles -> ms
    mean_inf_time = np.mean(inf_times) * 1e-6                 # ns -> ms
    print(f'Mean inference time (system clock):      {mean_inf_time:.3f} ms')
    print(f'Mean on-chip time (chip clock cycles):   {mean_inf_clk:.3f} ms')

### Full Model Benchmark

The loop above is clear, but it misses two things: power consumption, and a comparison
between mapping modes. `full_model_benchmark` from
[brainchip_utils/hardware_utils.py](../../../brainchip_utils/hardware_utils.py) runs
the same timed loop while also coordinating INA219 power measurement in a separate
process. The multiprocessing and power-meter wiring are non-trivial and not of
interest to most users — consult the source if you need the details.

It sweeps both mapping modes so the trade-off is visible:

- **`MapMode.Minimal`** — the fewest NPs the model needs, keeping power low.
- **`MapMode.AllNps`** — the model spread across more NPs. Power during inference
  rises slightly, and latency falls roughly in proportion.

The `num_sequences` check is worth watching: more than one sequence means part of the
model did not fit on the device and is running in software, which would make the
timings meaningless. This model maps to 6 Akida layers in a single pass, so it should
report one sequence.

In [ ]:
from brainchip_utils.hardware_utils import full_model_benchmark, get_mapping_stats
from brainchip_utils.plot_utils import plot_full_model_results

if device is not None:
    map_modes = ['Minimal', 'AllNps']
    POWER_REPEATS = 10
    full_results = {}
    for mm in map_modes:
        map_mode = getattr(akida.MapMode, mm)
        print(f'Running full-model benchmark (MapMode={mm}, {POWER_REPEATS} repeat(s))...')
        full_results[mm] = full_model_benchmark(
            akida_model, device, samples, map_mode=map_mode, repeats=POWER_REPEATS)

        # Re-map without hw_only to populate akida_model.sequences for stats
        akida_model.map(device, mode=map_mode)
        num_nps, num_passes, num_sequences = get_mapping_stats(akida_model)
        full_results[mm]['num_nps'] = num_nps
        full_results[mm]['num_passes'] = num_passes
        print(f'  Mapping: {num_nps} NP(s), {num_passes} pass(es), {num_sequences} sequence(s)')
        if num_sequences > 1:
            print('WARNING: model not completely mapped to hardware')

The plot below shows one column per map mode: a power trace (if a power meter was
connected) and the hardware mapping layout.

In [ ]:
if device is not None:
    plot_full_model_results(full_results, akida_model, device,
                            model_name='arrhythmia_classification')

### Per-Layer Benchmark

Full-model timing tells us the total cost but not where the time goes.
`per_layer_benchmark` from
[brainchip_utils/hardware_utils.py](../../../brainchip_utils/hardware_utils.py)
reconstructs latency layer by layer, by running cumulative sub-models and differencing
the results.

Several factors determine a layer's cost: the volume of inputs it receives and its
number of filters, the layer type and kernel size, and how many NPs it is spread
over. On Akida there is one more, and it is often the strongest: because the hardware
processes *events* — non-zero activations — a layer's cost is proportional to its
**input** sparsity. A layer receiving 90% sparse inputs has far fewer events to
process than one receiving 10% sparse inputs, and takes correspondingly less time.

That is why the per-layer timings and the sparsity values measured earlier are
naturally correlated, and it is worth comparing the two panels of the plot directly.

In [ ]:
from brainchip_utils.hardware_utils import per_layer_benchmark
from brainchip_utils.plot_utils import plot_per_layer_results

if device is not None:
    # Map without hw_only so akida_model.sequences is populated for the plot
    akida_model.map(device, mode=akida.MapMode.Minimal)

    print(f'Running per-layer benchmark ({len(samples)} samples)...')
    per_layer_results = per_layer_benchmark(akida_model, device, samples,
                                            repeats=len(samples))

The plot stacks three panels: per-layer latency, input sparsity per layer, and the
hardware mapping. The inverse relationship between sparsity and latency is the direct
signature of the event-driven compute model — dense activations generate more events,
and more events mean more work for the hardware.

In [ ]:
if device is not None:
    plot_per_layer_results(per_layer_results, akida_model, sparsity_dict,
                           model_name='arrhythmia_classification')